# NFL Game Outcome Model
Predicts whether the **home team wins** an NFL game, using each team's recent form going into the game.

**Data:** `data/processed/team_games.csv` (one row per team per game, 2002-2014), built from the scraped `data/raw/nfl_game_logs.csv` by `src/build_matches.py`.

**Evaluation design**
- Every feature uses only games played *before* the game being predicted.
- The model setup is chosen on a validation period (2011-2012 seasons) and then scored **once** on the test period (2013-2014 seasons).
- Results are compared against the "always pick the home team" baseline.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

matches = pd.read_csv("../data/processed/team_games.csv", index_col=0)
matches["date"] = pd.to_datetime(matches["date"])
print(matches.shape, matches["date"].min().date(), "to", matches["date"].max().date())

## 1. Game-level features
For each team, take the rolling average of its last `w` games (the current game is excluded with `closed="left"`).
Then build **one row per game**: the home team's rolling stats **minus** the away team's. This puts both teams' recent form in the same row, which a per-team row cannot do.

In [ ]:
STATS = ["avg_gain", "net_yards_passing", "net_yards_rushing", "pRate", "Y/P", "TO", "third_success_rate",
         "time_of_possession", "score", "first_downs", "penalties", "PenYds", "total_plays"]
COLS = [f"{p}_{s}" for p in ("team", "opp") for s in STATS]   # team_* = what the team did, opp_* = what it allowed


def game_table(w):
    """Home-minus-away rolling features, one row per game. Returns X, y (1 = home win), meta."""
    parts = []
    for _, g in matches.groupby("team"):
        g = g.sort_values("date").copy()
        g[COLS] = g[COLS].rolling(w, closed="left", min_periods=w).mean()
        parts.append(g)
    r = pd.concat(parts)
    home = r[r.venue == "Home"]
    away = r[r.venue == "Away"][["date", "team"] + COLS].rename(columns={"team": "opponent"})
    g = home.merge(away, on=["date", "opponent"], suffixes=("_h", "_a"))
    g = g.dropna(subset=[c + s for c in COLS for s in ("_h", "_a")])
    X = pd.DataFrame({c + "_diff": g[c + "_h"].values - g[c + "_a"].values for c in COLS})
    X["point_diff_diff"] = X["team_score_diff"] - X["opp_score_diff"]   # avg points scored minus allowed
    y = (g["result"] == "W").astype(int).reset_index(drop=True)
    meta = g[["date", "season"]].reset_index(drop=True)
    return X, y, meta


X, y, meta = game_table(10)
print(X.shape)
X.head(3)

## 2. Train / validation / test split (by date, never shuffled)
- Train: before Aug 2011  
- Validation: 2011-2012 seasons (used to pick the setup)  
- Test: 2013-2014 seasons (scored once at the end)

In [ ]:
TRAIN_END, VAL_END = pd.Timestamp("2011-08-01"), pd.Timestamp("2013-08-01")


def split(meta):
    tr = meta.date < TRAIN_END
    va = (meta.date >= TRAIN_END) & (meta.date < VAL_END)
    te = meta.date >= VAL_END
    return tr, va, te


tr, va, te = split(meta)
print(f"validation games: {va.sum()}   test games: {te.sum()}")
print(f"home team won {y[va].mean():.1%} of validation games and {y[te].mean():.1%} of test games")

## 3. Pick a setup on the validation period
Windows of 3, 6, 10 and 16 games; all 26 stat differences or just the point differential; logistic regression at three regularization strengths, plus a shallow random forest.

In [ ]:
def models():
    for C in (0.01, 0.1, 1.0):
        yield f"LogReg C={C}", lambda C=C: make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))
    for d, leaf in ((3, 20), (5, 30)):
        yield f"RandomForest depth={d} leaf={leaf}", lambda d=d, leaf=leaf: RandomForestClassifier(
            n_estimators=300, max_depth=d, min_samples_leaf=leaf, random_state=1)


rows = []
for w in (3, 6, 10, 16):
    Xw, yw, mw = game_table(w)
    trw, vaw, tew = split(mw)
    for fs in ("all 26 diffs", "points only"):
        cols = ["point_diff_diff"] if fs == "points only" else [c for c in Xw.columns if c != "point_diff_diff"]
        for name, make in models():
            if fs == "points only" and name.startswith("RandomForest"):
                continue
            fit = make().fit(Xw.loc[trw, cols], yw[trw])
            rows.append(dict(window=w, features=fs, model=name,
                             val_acc=(fit.predict(Xw.loc[vaw, cols]) == yw[vaw]).mean(),
                             test_acc=None, cols=cols))
results = pd.DataFrame(rows)
print(results.sort_values("val_acc", ascending=False).head(8)[["window", "features", "model", "val_acc"]].round(4).to_string(index=False))

## 4. Score the chosen setup once on the test period
The final model is trained on everything before the test period (2002-2012), the same training years as the original project.

In [ ]:
best = results.sort_values("val_acc", ascending=False).iloc[0]
Xb, yb, mb = game_table(best.window)
_, _, teb = split(mb)
trb_all = mb.date < VAL_END
make = dict(models())[best.model]
final = make().fit(Xb.loc[trb_all, best.cols], yb[trb_all])
pred = final.predict(Xb.loc[teb, best.cols])

n = int(teb.sum())
acc = (pred == yb[teb]).mean()
home_base = yb[teb].mean()
se = np.sqrt(acc * (1 - acc) / n)
print(f"Chosen on validation: window={best.window}, {best.features}, {best.model}")
print(f"Test games: {n}")
print(f"Always pick home team: {home_base:.1%}")
print(f"Model accuracy:        {acc:.1%}   (95% interval about {acc - 1.96 * se:.1%} to {acc + 1.96 * se:.1%})")

## 5. How every configuration did
The chart shows validation accuracy against test accuracy for all configurations. The point is to see the spread, not just the one picked: with ~500 test games, differences of a few points are within noise.

In [ ]:
for i, r in results.iterrows():
    Xw, yw, mw = game_table(r.window)
    trw, vaw, tew = split(mw)
    fit = dict(models())[r.model]().fit(Xw.loc[mw.date < VAL_END, r.cols], yw[mw.date < VAL_END])
    results.loc[i, "test_acc"] = (fit.predict(Xw.loc[tew, r.cols]) == yw[tew]).mean()

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(results.val_acc * 100, results.test_acc.astype(float) * 100, alpha=0.7)
ax.axhline(home_base * 100, color="gray", linestyle="--", label=f"always home ({home_base:.1%})")
ax.set_xlabel("Validation accuracy, 2011-2012 (%)")
ax.set_ylabel("Test accuracy, 2013-2014 (%)")
ax.set_title("All configurations tried")
ax.legend()
plt.tight_layout()
plt.show()
t = results.test_acc.astype(float)
print(f"Test accuracy across all {len(t)} configurations: min {t.min():.1%}, median {t.median():.1%}, max {t.max():.1%}")

## 6. Which features matter?
Standardized logistic-regression coefficients for the chosen model (larger magnitude = more influence on predicting a home win).

In [ ]:
if best.model.startswith("LogReg"):
    coefs = pd.Series(final[-1].coef_[0], index=best.cols).sort_values(key=abs, ascending=False).head(10)
    print(coefs.round(3).to_string())

## Notes and limits
- About 500 test games means a standard error of roughly 2 percentage points, so the edge over the home baseline is modest, not proven.
- Only team-level game stats are used (no injuries, quarterbacks, weather or betting lines).
- The first games of 2002 are dropped because there is no prior history to average.
- Setup selection used only the validation seasons; the test seasons were scored once for the chosen model. The all-configuration chart in section 5 is for context and was not used for selection.